# Self-Attention Variants - Consolidated Index

**Date**: 2026-05-31  

**Objective**: Provide a single lightweight index notebook to discover, validate, and run the separate attention-variant notebooks.

## 1) Imports and Setup

In [1]:
from __future__ import annotations



import importlib.util

import os

import random

import subprocess

from dataclasses import dataclass

from pathlib import Path



import numpy as np



SEED: int = 42

random.seed(SEED)

np.random.seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)





def parse_use_gpu_flag(raw_value: str) -> bool:

    normalized = raw_value.strip().lower()

    return normalized not in {"0", "false", "no", "off"}





def load_runtime_env() -> None:

    env_files = [

        Path("configs/runtime.env"),

        Path("configs/runtime.env.example"),

        Path("../configs/runtime.env"),

        Path("../configs/runtime.env.example"),

    ]

    for env_file in env_files:

        if env_file.exists():

            for line in env_file.read_text(encoding="utf-8").splitlines():

                line = line.strip()

                if not line or line.startswith("#") or "=" not in line:

                    continue

                key, value = line.split("=", 1)

                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

            break





load_runtime_env()

USE_GPU = parse_use_gpu_flag(os.getenv("USE_GPU", "1"))



_torch_cuda = False

_tf_gpu = False

if importlib.util.find_spec("torch") is not None:

    import torch



    _torch_cuda = torch.cuda.is_available()

if importlib.util.find_spec("tensorflow") is not None:

    import tensorflow as tf



    tf.random.set_seed(SEED)

    _tf_gpu = bool(tf.config.list_physical_devices("GPU"))



runtime_device = "cuda" if USE_GPU and (_torch_cuda or _tf_gpu) else "cpu"

print(f"USE_GPU={int(USE_GPU)} | runtime_device={runtime_device}")


USE_GPU=1 | runtime_device=cuda


## 2) Configuration and Constants

In [2]:
@dataclass(frozen=True)

class IndexConfig:

    base_dir: Path = Path("self-attention-variants")

    tested_dir_name: str = "tested"

    expected_files: tuple[str, ...] = (

        "01-scaled-dot-product-attention.ipynb",

        "02-causal-attention.ipynb",

        "03-local-window-attention.ipynb",

        "04-multi-head-attention.ipynb",

    )





CFG = IndexConfig()

CFG


IndexConfig(base_dir=PosixPath('self-attention-variants'), tested_dir_name='tested', expected_files=('01-scaled-dot-product-attention.ipynb', '02-causal-attention.ipynb', '03-local-window-attention.ipynb', '04-multi-head-attention.ipynb'))

## 3) Index Discovery and Status

In [3]:
def discover_variant_notebooks(cfg: IndexConfig) -> list[dict[str, str]]:

    rows: list[dict[str, str]] = []

    tested_dir = cfg.base_dir / cfg.tested_dir_name

    for file_name in cfg.expected_files:

        source_path = cfg.base_dir / file_name

        tested_path = tested_dir / file_name.replace(".ipynb", "_tested.ipynb")

        rows.append(

            {

                "file": file_name,

                "source_exists": "yes" if source_path.exists() else "no",

                "tested_exists": "yes" if tested_path.exists() else "no",

                "source_path": str(source_path),

                "tested_path": str(tested_path),

            }

        )

    return rows





rows = discover_variant_notebooks(CFG)

rows


[{'file': '01-scaled-dot-product-attention.ipynb',
  'source_exists': 'no',
  'tested_exists': 'no',
  'source_path': 'self-attention-variants/01-scaled-dot-product-attention.ipynb',
  'tested_path': 'self-attention-variants/tested/01-scaled-dot-product-attention_tested.ipynb'},
 {'file': '02-causal-attention.ipynb',
  'source_exists': 'no',
  'tested_exists': 'no',
  'source_path': 'self-attention-variants/02-causal-attention.ipynb',
  'tested_path': 'self-attention-variants/tested/02-causal-attention_tested.ipynb'},
 {'file': '03-local-window-attention.ipynb',
  'source_exists': 'no',
  'tested_exists': 'no',
  'source_path': 'self-attention-variants/03-local-window-attention.ipynb',
  'tested_path': 'self-attention-variants/tested/03-local-window-attention_tested.ipynb'},
 {'file': '04-multi-head-attention.ipynb',
  'source_exists': 'no',
  'tested_exists': 'no',
  'source_path': 'self-attention-variants/04-multi-head-attention.ipynb',
  'tested_path': 'self-attention-variants/teste

## 4) Optional WSL Execution and Verification

In [4]:
def build_wsl_nbconvert_command(file_name: str) -> str:

    output_name = file_name.replace(".ipynb", "_tested.ipynb")

    return (

        "wsl -d Ubuntu -- bash -lc "

        "\"source ~/.bashrc_dev; "

        "source ~/APPS_VENV/python_venv/run_3_14_2/bin/activate; "

        "cd /mnt/c/DEV/PROJECTS/ML_BASICS/machine-learning-labs; "

        f"mkdir -p self-attention-variants/tested; "

        f"jupyter nbconvert --to notebook --execute self-attention-variants/{file_name} "

        f"--output-dir self-attention-variants/tested --output {output_name}\""

    )





run_commands = [build_wsl_nbconvert_command(row["file"]) for row in rows]

run_commands


['wsl -d Ubuntu -- bash -lc "source ~/.bashrc_dev; source ~/APPS_VENV/python_venv/run_3_14_2/bin/activate; cd /mnt/c/DEV/PROJECTS/ML_BASICS/machine-learning-labs; mkdir -p self-attention-variants/tested; jupyter nbconvert --to notebook --execute self-attention-variants/01-scaled-dot-product-attention.ipynb --output-dir self-attention-variants/tested --output 01-scaled-dot-product-attention_tested.ipynb"',
 'wsl -d Ubuntu -- bash -lc "source ~/.bashrc_dev; source ~/APPS_VENV/python_venv/run_3_14_2/bin/activate; cd /mnt/c/DEV/PROJECTS/ML_BASICS/machine-learning-labs; mkdir -p self-attention-variants/tested; jupyter nbconvert --to notebook --execute self-attention-variants/02-causal-attention.ipynb --output-dir self-attention-variants/tested --output 02-causal-attention_tested.ipynb"',
 'wsl -d Ubuntu -- bash -lc "source ~/.bashrc_dev; source ~/APPS_VENV/python_venv/run_3_14_2/bin/activate; cd /mnt/c/DEV/PROJECTS/ML_BASICS/machine-learning-labs; mkdir -p self-attention-variants/tested; ju

In [5]:
RUN_ALL_VARIANTS = False




def execute_variant_notebooks(commands: list[str], run_enabled: bool) -> list[dict[str, str]]:


    if not run_enabled:


        print("RUN_ALL_VARIANTS is False; skipping execution.")


        return []




    results: list[dict[str, str]] = []


    for idx, command in enumerate(commands, start=1):


        print(f"Running {idx}/{len(commands)}")


        completed = subprocess.run(


            command,


            shell=True,


            capture_output=True,


            text=True,


        )


        results.append(


            {


                "order": str(idx),


                "return_code": str(completed.returncode),


                "stdout_tail": "\n".join(completed.stdout.splitlines()[-5:]),


                "stderr_tail": "\n".join(completed.stderr.splitlines()[-5:]),


            }


        )


    return results






execution_results = execute_variant_notebooks(run_commands, RUN_ALL_VARIANTS)


execution_results[:1] if execution_results else execution_results

RUN_ALL_VARIANTS is False; skipping execution.


[]

## 4.1) Optional Execute-All Runner (Safe Toggle)


Set `RUN_ALL_VARIANTS = True` only when you want this notebook to execute all variant notebooks automatically.

## 5) Summary and Next Steps

In [6]:
source_ok = sum(1 for row in rows if row["source_exists"] == "yes")

tested_ok = sum(1 for row in rows if row["tested_exists"] == "yes")



summary = {

    "expected_notebooks": len(CFG.expected_files),

    "source_notebooks_found": source_ok,

    "tested_notebooks_found": tested_ok,

    "source_coverage": source_ok / len(CFG.expected_files),

    "tested_coverage": tested_ok / len(CFG.expected_files),

}

summary


{'expected_notebooks': 4,
 'source_notebooks_found': 0,
 'tested_notebooks_found': 0,
 'source_coverage': 0.0,
 'tested_coverage': 0.0}